# lift runner: Kaggle T4 x2 (backup GPU)

Reported runs use a Colab A100 in bf16 (`colab/README.md`). This notebook is the backup runner; `kaggle/README.md` has the one-time setup.

1. **Settings.** Accelerator *GPU T4 x2*, Internet *on*. Attach the Kaggle secrets `HF_TOKEN` (a write token), `GITHUB_TOKEN` and `LIFT_RUNS_REPO`, plus `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and `LIFT_JUDGE_MAX_USD` for frontier-judge jobs.
2. **Cell 1, once per session.** Set `BRANCH` to a ref with both P01 and P02 (`main` once both are merged; before that, an integration branch with `p01-data` merged into `p02-harness`, pushed), then run it. It reads the secrets into `SECRETS` without printing them, clones the repo and installs it with `kaggle/bootstrap.sh`. The secrets never go into the environment: git, pip and uv never see them, and cell 2 gives them only to the `lift` commands. Rerun it after a commit to `BRANCH`, such as the pin step after Gate 0.
3. **Cell 2, the job.** Paste the `lift` lines Claude Code printed into `JOB` and run the cell. It first checks that the runs repo is private and that `HF_TOKEN` can write to it (`lift sync check --write`). Lines run in order and the first failing line stops the job. A `lift generate`, `lift judge` or `lift run-dev` line that exits 3 does not: it is done but skipped some cells, each listed with its remedy, and the next lines still run. Nor does a `lift sync` line that exits 3 (never retried): it is done, but some files conflicted (a push's local copies go under `conflicts/` in the runs repo, a pull keeps them) or some judgements or AuditKIT records do not pair with their cell, and the next lines still run, so the `lift restart` line its summary names for those, put right after `lift sync pull`, runs too. Outputs are pushed to the runs repo after every line and once more at the end, even after a failure or an interrupt. A `lift sync` command that exits 1 is retried (3 attempts over about 3 minutes). One that runs past 30 minutes is stopped: a push is then tried again (up to about 1.5 hours in all), and a stopped pull, check or status ends the job (the final push still runs). The final push gets one try after a stall when no line but `lift sync` has run in this kernel and `JOB` holds no `lift sync push` line: this kernel made nothing to push, and what an earlier kernel left (a kernel restart) goes up with `JOB` set to `lift sync push`. An exit 2 is a configuration problem that no retry fixes (a token that cannot write, a missing or public runs repo, a `BRANCH` without `lift sync`) or a `lift sync` line's own bad arguments: no further line runs, and after the check's or a plain `lift sync push`'s exit 2 no push either, while after another sync line's (a push with more words included) the final push still runs. Apart from that, a push never stops the job. A line that prints nothing for 45 minutes is stopped and fails. A running command dies with the kernel, but interrupt cell 2 before restarting the kernel: only the interrupt runs the final push. The cell raises at the end if anything failed, an exit 3 included. For anything over 20 minutes, use **Save Version > Save & Run All (Commit)**.

**The T4 rule.** A T4 has no bf16, so every `generate`, `judge` and `run-dev` line takes `--dtype fp16`. A T4 number counts only while Gate 0's fp16-vs-bf16 check (`lift smoke --compare-dtype fp16`, run here) and the bf16 Gate 0 it compared against (`lift smoke`, run on Colab) both pass for the same Global revision, and fp16 numbers are never paired with bf16 ones. Over a variant Colab started, leave its bf16 cells out with `--sets` and `--langs`, and never restart them from here (`kaggle/README.md`, Jobs on a T4 x2). The generator runs on `cuda:0` and the judge on `cuda:1`.

**Privacy.** Kaggle saves `/kaggle/working` and the printed output with every notebook version. The clone, the venv and every data, output and validation file live under `/kaggle/temp/lift`; only `lift_env` (the interpreter path) goes in `/kaggle/working`. Kaggle describes `/kaggle/temp` as not saved, which is unverified here: check that the first committed version's Output holds only `lift_env`. Nothing here prints a secret or a model output. Keep this notebook private.

In [ ]:
import base64
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

# A ref with both P01 and P02: main once both are merged, until then an integration branch with p01-data
# merged into p02-harness, pushed. The clone is taken when this cell runs: rerun it after a commit to
# BRANCH, such as the pin step after Gate 0 (kaggle/README.md).
BRANCH = "main"
REPO_URL = "https://github.com/Knull/lost-in-finetuning.git"
# Kaggle saves /kaggle/working with every notebook version, so the clone, the venv and every data,
# output and validation file live under /kaggle/temp, which Kaggle describes as not saved (unverified).
ROOT = "/kaggle/temp/lift"
REPO = f"{ROOT}/repo"
LIFT_DIRS = {"LIFT_DATA_DIR": f"{ROOT}/data", "LIFT_OUTPUT_DIR": f"{ROOT}/outputs",
             "LIFT_VALIDATION_DIR": f"{ROOT}/validation"}
REQUIRED = ["HF_TOKEN", "GITHUB_TOKEN", "LIFT_RUNS_REPO"]
OPTIONAL = ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "LIFT_JUDGE_MAX_USD"]
# No child process inherits these. The secrets go only to the lift commands (cell 2), and with Kaggle's
# client tokens any process could fetch every attached secret: pip and uv run third-party build code.
HIDDEN = [*REQUIRED, *OPTIONAL, "KAGGLE_USER_SECRETS_TOKEN", "KAGGLE_IAP_TOKEN"]


def load_secrets(get_secret):
    """Return the attached Kaggle secrets as a dict; they never go into os.environ (see HIDDEN).
    Unattached or empty optional ones are skipped, and no value is ever printed."""
    secrets = {}
    for name in REQUIRED + OPTIONAL:
        try:
            value = get_secret(name).strip()
        except Exception:
            continue
        if value:
            secrets[name] = value
    missing = [name for name in REQUIRED if name not in secrets]
    if missing:
        raise RuntimeError(f"attach these Kaggle secrets to the notebook (Add-ons > Secrets): {missing}")
    return secrets


def child_env(extra=()):
    """The environment of every child process: os.environ without the HIDDEN names, plus extra."""
    env = {name: value for name, value in os.environ.items() if name not in HIDDEN}
    env.update(extra)
    return env


def clone(token, run=subprocess.run):
    """Shallow-clone BRANCH into REPO. The token travels only in an HTTP header set through
    GIT_CONFIG_* variables: never in the URL, the command line, .git/config or an error message.
    The runtime's own GIT_CONFIG_PARAMETERS and GIT_CONFIG_COUNT/KEY_n/VALUE_n are left out of the
    clone (they would be read with this header: a second AUTHORIZATION header, a URL rewrite), not
    out of os.environ or bootstrap.sh."""
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {name: value for name, value in child_env().items()
           if name not in ("GIT_CONFIG_COUNT", "GIT_CONFIG_PARAMETERS")
           and not name.startswith(("GIT_CONFIG_KEY_", "GIT_CONFIG_VALUE_"))}
    env.update({"GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
                "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"})
    shutil.rmtree(REPO, ignore_errors=True)
    result = run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO],
                 env=env, capture_output=True, text=True)
    if result.returncode != 0:
        detail = result.stderr.replace(token, "[redacted]").replace(auth, "[redacted]").strip()
        raise RuntimeError(f"git clone of {BRANCH!r} failed (exit {result.returncode}): {detail}")


SECRETS = load_secrets(UserSecretsClient().get_secret)
os.environ.update(LIFT_DIRS)
clone(SECRETS["GITHUB_TOKEN"])
if subprocess.run(["bash", f"{REPO}/kaggle/bootstrap.sh"], cwd=ROOT, env=child_env()).returncode != 0:
    raise RuntimeError("kaggle/bootstrap.sh failed: see its output above")

In [ ]:
import codecs
import os
import shlex
import signal
import subprocess
import sys
import threading
import time
from shutil import which

# One command per line, as Claude Code printed them; lines starting with # are skipped. There is no
# need to end with "lift sync push": the outputs are pushed after every line and once more at the end.
JOB = """
lift sync pull
lift info
# paste the stage's commands here, one per line (on a T4, generate/judge/run-dev take --dtype fp16)
"""
# The watchdog, as in colab/job.py. A line that prints nothing for MAX_SILENT_MINUTES is hung: it is
# stopped and fails. A lift sync command running past PUSH_LIMIT_MINUTES is stopped, and a push is tried
# again: huggingface_hub sets no network timeout, so one stalled read would hold the session until
# Kaggle's 12-hour limit. Stopping is SIGINT, so lift can close its files, then SIGKILL to the command's
# whole process group KILL_GRACE_SECONDS later. Raise MAX_SILENT_MINUTES for a step silent longer by design.
MAX_SILENT_MINUTES = 45
PUSH_LIMIT_MINUTES = 30
KILL_GRACE_SECONDS = 30
TICK_SECONDS = 5  # how often the watchdog looks
# lift sync exits 0 when done; 1 on an error a retry may fix (not everything was transferred); 2 on a
# problem no retry fixes (a token that cannot write, the runs repo missing or public, a BRANCH without
# lift sync, or the line's own arguments), after which no line runs; 3 when it is done but some files
# conflicted (push: their local copies went under conflicts/ in the runs repo; pull: the local copies were
# kept) or judgements or AuditKIT records do not pair with their cell (the summary names the lift restart
# line that sets them aside, a JOB line right after lift sync pull): reported, never retried, and the next
# lines run. A committed version starts on an empty disk, so its pull meets every such file in the runs repo.
RETRY_WAITS = (60, 120)  # seconds before the 2nd and 3rd try of a sync that exits 1 (or a stopped push)
PREFLIGHT = "lift sync check --write"  # before any line: the runs repo is private and HF_TOKEN can write
PUSH = "lift sync push"  # the runner's own push: a checkpoint after each line but a sync line, and the last
# lift generate, judge and run-dev exit 3 when they are done but skipped some cells (stale, made in another
# precision, judged by another judge) or had an AuditKIT record refused, each listed with its remedy: the
# job reports it and goes on with the next lines. Any other non-zero exit of a line stops the job.
SKIPPED_CELLS = 3
CELL_COMMANDS = ("generate", "judge", "run-dev")
# Whether a line other than lift sync has started in this kernel. Until one has, the kernel has made nothing
# to push, so a stalled final push gets one try, unless JOB holds a lift sync push line: such a job is there
# to send what an earlier kernel left (a kernel restart). Rerunning this cell keeps it.
LINES_RAN = globals().get("LINES_RAN", False)
KILL = getattr(signal, "SIGKILL", signal.SIGTERM)  # no SIGKILL on Windows, where only the tests' fakes run
# A command must not outlive the kernel. Its own process group, which the watchdog signals, takes it out
# of the kernel's group, which a forced kernel restart or shutdown kills; so on Linux it starts through
# this shim (python -S -c PDEATHSIG_SHIM <kernel pid> <command...>). The shim asks for SIGKILL when the
# kernel dies (PR_SET_PDEATHSIG, kept across exec; strictly, when the thread that started it ends, and
# that thread waits for the command), undoes what Python's own startup did to the signal dispositions,
# and execs the command: its argv, environment and exit code stay its own. Not preexec_fn, which is
# unsafe with the reader threads running.
PDEATHSIG_SHIM = """\
import sys
sys.path = [p for p in sys.path if p]  # nothing imported from the cwd
import os, signal
try:
    import ctypes
    ctypes.CDLL(None).prctl(1, *map(ctypes.c_ulong, (signal.SIGKILL, 0, 0, 0)))  # PR_SET_PDEATHSIG
except Exception:  # no prctl after all: the command runs untied
    pass
if os.getppid() != int(sys.argv[1]):  # the kernel died before prctl took hold
    os.kill(os.getpid(), signal.SIGKILL)
if signal.getsignal(signal.SIGINT) is signal.default_int_handler:
    signal.signal(signal.SIGINT, signal.SIG_DFL)
for name in ("SIGPIPE", "SIGXFZ", "SIGXFSZ"):  # as subprocess's restore_signals leaves them
    if hasattr(signal, name):
        signal.signal(getattr(signal, name), signal.SIG_DFL)
os.execvp(sys.argv[2], sys.argv[2:])
"""


def prctl_available():
    """Whether commands can start through PDEATHSIG_SHIM: Linux, with prctl in the C library."""
    if not sys.platform.startswith("linux") or not sys.executable:
        return False
    try:
        import ctypes
        return hasattr(ctypes.CDLL(None), "prctl")
    except (ImportError, OSError):
        return False


PDEATHSIG = prctl_available()


def tied_to_kernel(argv, env):
    """argv started through PDEATHSIG_SHIM where prctl exists. A command that cannot be found starts as
    it is, so Popen raises for it as it always did."""
    if not PDEATHSIG or which(argv[0], path=env.get("PATH", os.defpath)) is None:
        return argv
    return [sys.executable, "-S", "-c", PDEATHSIG_SHIM, str(os.getpid()), *argv]


def hms(seconds):
    s = int(seconds)
    return f"{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}"


def signal_group(proc, sig):
    """Send sig to the command's whole process group: a child of lift may hold the output pipe open."""
    try:
        if hasattr(os, "killpg"):
            os.killpg(proc.pid, sig)
        else:
            proc.send_signal(sig)
    except (OSError, ValueError):  # it has ended meanwhile
        pass


def stop(proc):
    """Stop a command now: SIGINT, then SIGKILL to its group KILL_GRACE_SECONDS later, even when another
    interrupt arrives meanwhile."""
    try:
        signal_group(proc, signal.SIGINT)
        proc.wait(timeout=KILL_GRACE_SECONDS)
    except subprocess.TimeoutExpired:
        pass
    finally:
        signal_group(proc, KILL)
        try:
            proc.wait(timeout=KILL_GRACE_SECONDS)
        except subprocess.TimeoutExpired:
            print(f"!! process {proc.pid} outlived SIGKILL", flush=True)


def forward(stream, last, clock, masks=()):
    """Copy a command's output to the cell as it comes, and note when it last printed. The decoder is
    incremental, so a character split across two reads survives. Kaggle saves the printed output with
    every version, so each value in ``masks`` (the attached secrets) is printed as ***: the output goes
    out a whole line at a time ("\n" or "\r"), which no secret spans. A megabyte with no line end is
    printed masked, all but the tail a secret could still be starting in."""
    decode = codecs.getincrementaldecoder("utf-8")(errors="replace").decode
    read = getattr(stream, "read1", stream.read)
    keep = max(map(len, masks), default=1) - 1
    pending = ""

    def scrub(text):
        for secret in masks:
            text = text.replace(secret, "***")
        return text

    try:
        for chunk in iter(lambda: read(1 << 16), b""):
            last[0] = clock()
            pending += decode(chunk)
            cut = max(pending.rfind("\n"), pending.rfind("\r")) + 1
            if not cut and len(pending) > 1 << 20:
                pending = scrub(pending)
                cut = len(pending) - keep
            if cut > 0:
                print(scrub(pending[:cut]), end="", flush=True)
                pending = pending[cut:]
    except (OSError, ValueError):  # the pipe was closed under the reader
        pass
    print(scrub(pending + decode(b"", final=True)), end="", flush=True)


def overdue(sync, ran, silent):
    """Why the watchdog stops a command now, or None."""
    if sync and ran > PUSH_LIMIT_MINUTES * 60:
        return f"it has run for {hms(ran)} (PUSH_LIMIT_MINUTES = {PUSH_LIMIT_MINUTES})"
    if not sync and silent > MAX_SILENT_MINUTES * 60:
        return f"no output for {hms(silent)} (MAX_SILENT_MINUTES = {MAX_SILENT_MINUTES})"
    return None


def run_command(argv, line, *, sync, cwd, env, popen, clock, masks=()):
    """Run argv to its end in its own process group, tied to the kernel (tied_to_kernel), its output
    copied to the cell as it comes, under the watchdog. An interrupt stops the command the same way and
    is raised again. Returns (exit code, why the watchdog stopped it or None); a command ended by signal
    N gives 128 + N."""
    proc = popen(tied_to_kernel(argv, env), cwd=cwd, env=env, stdin=subprocess.DEVNULL,
                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
    started = clock()
    last = [started]
    reader = threading.Thread(target=forward, args=(proc.stdout, last, clock, masks), daemon=True)
    reader.start()
    why = stopped = None
    try:
        while True:
            try:
                rc = proc.wait(timeout=TICK_SECONDS)
                break
            except subprocess.TimeoutExpired:
                now = clock()
            if why is None:
                why = overdue(sync, now - started, now - last[0])
                if why:
                    print(f"!! stopping {line!r}: {why}", flush=True)
                    stopped = now
                    signal_group(proc, signal.SIGINT)
            elif now - stopped >= KILL_GRACE_SECONDS:
                signal_group(proc, KILL)
    except BaseException:
        stop(proc)
        raise
    finally:
        if why:
            signal_group(proc, KILL)  # whatever of a stopped command's group outlived it (GPU memory)
        reader.join(KILL_GRACE_SECONDS)
    return (rc if rc >= 0 else 128 - rc), why


def run_job(job, lift_py, *, cwd, secrets, popen=subprocess.Popen, sleep=time.sleep, clock=time.monotonic):
    """Check the runs repo (PREFLIGHT), then run each line in order under the watchdog and stop at the
    first failing line that is not a push. A CELL_COMMANDS line that exits SKIPPED_CELLS is done but
    skipped some cells, and a lift sync line that exits 3 is done but met conflicts or unpaired files:
    each is reported, and the job goes on. After each line but a lift sync line that succeeds or only
    skipped cells, push the outputs to the runs repo as a checkpoint, and push once more in finally,
    even after a failure or an interrupt. A lift sync command that exits 1 (a push also when stopped,
    see sync) is tried again after RETRY_WAITS. An exit 2 is never retried, and no line runs after it.
    After the preflight's or a plain PUSH's, no push runs either: it would fail the same way. A JOB
    sync line's may come from its own arguments, so the final push still runs. Any other failed push
    is reported and the job goes on: the next push sends whatever is still missing. Only the lift
    commands get the secrets, and output lines are printed with them masked. Raise at the end if
    anything did not succeed (an exit 3 included), so a committed version shows as failed."""
    problems = []
    ended = []  # set when the preflight or a plain PUSH exits 2: no other line or sync command runs
    masks = sorted({value for value in secrets.values() if len(value) >= 8}, key=len, reverse=True)
    # A JOB with a lift sync push line is there to send outputs, an earlier kernel's too (kaggle/README.md),
    # so its final push is tried again after a stall even in a kernel where no other line has run.
    pushing = any(line.split()[:3] == PUSH.split() for line in job.splitlines())

    def report(problem):
        problems.append(problem)
        print("!!", problem, flush=True)

    def call(line):
        words = shlex.split(line)
        cmd, extra = words, {}
        if words[0] == "lift":
            cmd, extra = [lift_py, "-m", "lift", *words[1:]], secrets
        print(">>", line, flush=True)
        env = child_env({**extra, "PYTHONUNBUFFERED": "1"})
        return run_command(cmd, line, sync=words[:2] == ["lift", "sync"], cwd=cwd, env=env, popen=popen,
                           clock=clock, masks=masks)

    def sync(line, what, *, preflight=False, final=False):
        """A lift sync command, tried again after RETRY_WAITS while it exits 1, and a push also when the
        watchdog stopped it (a stalled upload), except the final push when no other line has run in this
        kernel (LINES_RAN) and JOB holds no push line (pushing): this kernel made nothing to send, and
        what an earlier kernel left goes up with a JOB of lift sync push. A stopped pull, check or
        status is not tried again: it already held the GPUs for PUSH_LIMIT_MINUTES, and another try would
        most likely stall as well. Returns its last exit code after reporting anything but a success; a
        failed one returns neither 0 nor 3, so it is never taken for a success or for a 3 (done: the job
        goes on). An exit 2 of the preflight or of a plain PUSH ends the job (ended): the final push is
        that very command. A JOB sync line's exit 2 (a pull, a status, a push with more words) may come
        from the line's own arguments, so the final push still runs."""
        words = shlex.split(line)
        push = words[2:3] == ["push"]
        stalls_retried = push and (LINES_RAN or pushing or not final)
        for attempt, wait in enumerate((*RETRY_WAITS, None), start=1):
            rc, why = call(line)
            failed = "was stopped" if why else "exited 1" if rc == 1 else None
            if not failed or wait is None or (why and not stalls_retried):
                break
            print(f"!! {what} {failed} (attempt {attempt}); trying again in {wait} s", flush=True)
            sleep(wait)
        if failed:
            if why and not stalls_retried:
                report(f"{what} was stopped and is not tried again: {why}" + (
                    "; no line but lift sync has run in this kernel, so it made nothing to push: to push "
                    "what an earlier kernel left (a kernel restart), run cell 2 with JOB set to lift sync "
                    "push once the Hub answers" if push else ""))
            else:
                report(f"{what} failed on all {attempt} attempts (the last {failed})")
            return 1 if rc in (0, 3) else rc  # a stopped command may have ended by itself meanwhile
        if rc == 2:
            last = preflight or words == PUSH.split()
            if last:
                ended.append(what)
            report(f"{what} exited 2: a configuration problem that no retry fixes (lift sync says which "
                   "above; kaggle/README.md, Troubleshooting). " + (
                       "The job ends here, with no further push" if last else
                       "No further line runs, but the final push still runs: this exit 2 may come from "
                       "the line's own arguments"))
        elif rc == 3:
            report(f"{what} exited 3 (conflicts or unpaired files, named in its summary above: " + (
                "conflicting local copies went under conflicts/ in the runs repo)" if push else
                "a conflict keeps the local copy, and unpaired files go aside with the lift restart line it "
                "names, a JOB line right after lift sync pull)"))
        elif rc:
            report(f"{what} exited {rc}")
        return rc

    def run_lines():
        """The lines in order, each but a sync line followed by a checkpoint push when it succeeded or
        only skipped some cells (SKIPPED_CELLS)."""
        global LINES_RAN
        for line in [line.strip() for line in job.splitlines()]:
            if not line or line.startswith("#"):
                continue
            words = shlex.split(line)
            if words[:2] == ["lift", "sync"]:
                rc = sync(line, repr(line))
                # An exit 2 stops the job, as does a pull, check or status that failed; a failed push does
                # not, and an exit 3 is done: the lift restart line a pull's summary names is a later line.
                if ended or rc == 2 or (rc not in (0, 3) and words[2:3] != ["push"]):
                    return
                continue
            LINES_RAN = True  # from here on, even if the line fails, there may be outputs to push
            rc, why = call(line)
            if not why and rc == SKIPPED_CELLS and words[:2] in [["lift", name] for name in CELL_COMMANDS]:
                report(f"{line!r} exited {rc}: done, but some cells were skipped or their AuditKIT record "
                       "was refused (lift lists each above, with its remedy); the next lines run")
            elif rc or why:
                report(f"{line!r} " + (f"was stopped: {why}" if why else f"exited {rc}"))
                return
            sync(PUSH, f"the checkpoint push after {line!r}")
            if ended:
                return

    try:
        if sync(PREFLIGHT, f"the preflight {PREFLIGHT!r}", preflight=True) == 0:
            run_lines()
    finally:
        if not ended:
            sync(PUSH, "the final push", final=True)
    if problems:
        raise RuntimeError("job FAILED: " + "; ".join(problems))
    print("job OK: every line succeeded and the outputs are pushed", flush=True)


if "SECRETS" not in globals() or "LIFT_OUTPUT_DIR" not in os.environ:
    raise RuntimeError("run the setup cell (cell 1) first: a restarted kernel has lost the secrets and the "
                       "LIFT_* dirs")
with open("/kaggle/working/lift_env") as f:
    LIFT_PY = f.read().split("=", 1)[1].strip()
# Relative paths in JOB resolve against ROOT, which holds data/, outputs/ and validation/.
run_job(JOB, LIFT_PY, cwd=ROOT, secrets=SECRETS)